Set True if you want to run a sweep

In [ ]:
do_sweep = True

Set systempath

In [ ]:
import sys
sys.path.append("../src")

Import everthing needed

In [ ]:
import pandas as pd
import numpy as np
import os
import wandb
import random
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from transforms.feature_engineering import add_all_features, filter_business_hours, entries_per_day_per_site
from transforms.feature_engineering import (
    CONTINUOUS_FEATURE_COLUMNS,
    CATEGORICAL_FEATURE_COLUMNS,
    CYCLIC_FEATURE_COLUMNS,
    TARGET_COLUMN
)
from evaluation.comp_metrics import evaluate_all_metrics

Set wandb key (don't push to repository)

In [ ]:
os.environ["WANDB_API_KEY"] = "3aaf9f796df65417b3f5f8560b43875171b55805"

Set seed

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Load the data

In [ ]:
df_train = pd.read_csv("../data/classification/classification-train.csv")
df_test = pd.read_csv("../data/classification/classification-test.csv")

In [ ]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

In [ ]:
df_train = add_all_features(df_train)

In [ ]:
df_train = filter_business_hours(df_train)

In [ ]:
df_train.columns

In [ ]:
ENTRIES_PER_DAY = entries_per_day_per_site(df_train)

In [ ]:
print(ENTRIES_PER_DAY)

In [ ]:
def prepare_data():
    """Prepare and return all data splits"""
    df_train_site_a = df_train[0:19345]
    df_train_site_b = df_train[19345:38690]
    df_train_site_c = df_train[38690:58035]

    X_continuous_site_a = df_train_site_a[CONTINUOUS_FEATURE_COLUMNS].values
    X_continuous_site_b = df_train_site_b[CONTINUOUS_FEATURE_COLUMNS].values
    X_continuous_site_c = df_train_site_c[CONTINUOUS_FEATURE_COLUMNS].values

    X_categorical_site_a = df_train_site_a[CATEGORICAL_FEATURE_COLUMNS].values
    X_categorical_site_b = df_train_site_b[CATEGORICAL_FEATURE_COLUMNS].values
    X_categorical_site_c = df_train_site_c[CATEGORICAL_FEATURE_COLUMNS].values

    X_cyclic_site_a = df_train_site_a[CYCLIC_FEATURE_COLUMNS].values
    X_cyclic_site_b = df_train_site_b[CYCLIC_FEATURE_COLUMNS].values
    X_cyclic_site_c = df_train_site_c[CYCLIC_FEATURE_COLUMNS].values

    y_site_a = df_train_site_a[TARGET_COLUMN].values.reshape(-1, 1)
    y_site_b = df_train_site_b[TARGET_COLUMN].values.reshape(-1, 1)
    y_site_c = df_train_site_c[TARGET_COLUMN].values.reshape(-1, 1)

    y_site_a_unscaled = y_site_a.copy()

    scaler_X_site_a = StandardScaler()
    scaler_X_site_b = StandardScaler()
    scaler_X_site_c = StandardScaler()

    scaler_y_site_a = StandardScaler()
    scaler_y_site_b = StandardScaler()
    scaler_y_site_c = StandardScaler()

    X_scaled_site_a = scaler_X_site_a.fit_transform(X_continuous_site_a)
    X_scaled_site_b = scaler_X_site_b.fit_transform(X_continuous_site_b)
    X_scaled_site_c = scaler_X_site_c.fit_transform(X_continuous_site_c)

    y_site_a = scaler_y_site_a.fit_transform(y_site_a)
    y_site_b = scaler_y_site_b.fit_transform(y_site_b)
    y_site_c = scaler_y_site_c.fit_transform(y_site_c)

    X_site_a = np.concatenate([X_scaled_site_a, X_categorical_site_a, X_cyclic_site_a], axis=1).astype(np.float32)
    X_site_b = np.concatenate([X_scaled_site_b, X_categorical_site_b, X_cyclic_site_b], axis=1).astype(np.float32)
    X_site_c = np.concatenate([X_scaled_site_c, X_categorical_site_c, X_cyclic_site_c], axis=1).astype(np.float32)

    y_site_a = y_site_a.astype(np.float32)
    y_site_b = y_site_b.astype(np.float32)
    y_site_c = y_site_c.astype(np.float32)

    # Remove NaN entries
    mask_site_a = np.ones(len(X_site_a), dtype=bool)
    mask_site_b = np.ones(len(X_site_b), dtype=bool)
    mask_site_c = np.ones(len(X_site_c), dtype=bool)

    mask_site_a[0:ENTRIES_PER_DAY] = False
    mask_site_b[0:ENTRIES_PER_DAY] = False
    mask_site_c[0:ENTRIES_PER_DAY] = False

    X_site_a = X_site_a[mask_site_a]
    X_site_b = X_site_b[mask_site_b]
    X_site_c = X_site_c[mask_site_c]

    y_site_a = y_site_a[mask_site_a]
    y_site_b = y_site_b[mask_site_b]
    y_site_c = y_site_c[mask_site_c]

    return {
        'X_site_a': X_site_a, 'X_site_b': X_site_b, 'X_site_c': X_site_c,
        'y_site_a': y_site_a, 'y_site_b': y_site_b, 'y_site_c': y_site_c,
        'scaler_X_site_a': scaler_X_site_a, 'scaler_X_site_b': scaler_X_site_b, 'scaler_X_site_c': scaler_X_site_c,
        'scaler_y_site_a': scaler_y_site_a, 'scaler_y_site_b': scaler_y_site_b, 'scaler_y_site_c': scaler_y_site_c,
        'y_site_a_unscaled': y_site_a_unscaled
    }

In [ ]:
def create_sequences(X, y, seq_length):
    sequences_X = []
    sequences_y = []
    
    for i in range(len(X) - seq_length):
        sequences_X.append(X[i:i+seq_length])
        sequences_y.append(y[i+seq_length - 1])

    print(f"Sequences X: {len(sequences_X)}, Sequences Y: {len(sequences_y)}")
    
    return np.array(sequences_X), np.array(sequences_y)

In [ ]:
class PowerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class TransformerRegressor(nn.Module):
    def __init__(self, input_size, d_model, nhead, num_layers, output_size, dropout=0.1, max_seq_len=500):
        super(TransformerRegressor, self).__init__()
        self.d_model = d_model
        
        # Input projection
        self.input_projection = nn.Linear(input_size, d_model)
        
        # FIXED: Dynamic positional encoding with larger max length
        self.pos_encoder = nn.Parameter(torch.randn(1, max_seq_len, d_model))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output layers
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(d_model, output_size)
    
    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        seq_len = x.size(1)
        
        # Project input to d_model dimension
        x = self.input_projection(x)  # (batch, seq_len, d_model)
        
        # Add positional encoding (now safely handles any seq_len <= max_seq_len)
        x = x + self.pos_encoder[:, :seq_len, :]
        
        # Transformer encoding
        x = self.transformer_encoder(x)  # (batch, seq_len, d_model)
        
        # Use the last token's output for prediction
        x = x[:, -1, :]  # (batch, d_model)
        
        # Dropout and final prediction
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [ ]:
def get_lr_with_warmup(epoch, base_lr, warmup_epochs):
    if warmup_epochs == 0 or epoch >= warmup_epochs:
        return base_lr
    else:
        return base_lr * (epoch + 1) / warmup_epochs

In [ ]:
def train_model(config=None):
    """
    Train the SimpleRNN model (replacement function).
    Changes implemented:
      - Use OneCycleLR for smooth LR (no step jumps)
      - Step scheduler each batch (smooth intra-epoch LR)
      - Ensure val DataLoader is chronological (shuffle=False, drop_last=False)
      - Configurable early stopping patience (use config.early_stopping_patience if provided)
      - Stable handling of penalty term and metric ordering
    """

    # -------------------------------
    # 1. Initialize Weights & Biases
    # -------------------------------
    wandb_run = wandb.init(
        project="AICOMP_Flextrack",
        entity="fabian-dubach-hochschule-luzern",
        config=config
    )
    config = wandb.config

    best_model_path = os.path.join(wandb_run.dir, "best_model.pt")

    print("\n" + "=" * 60)
    print("Starting run with config:")
    for k, v in dict(config).items():
        print(f"  {k}: {v}")
    print("=" * 60 + "\n")

    # -------------------------------
    # 2. Load + preprocess data
    # -------------------------------
    data = prepare_data()

    # Create sequences per site
    X_a, y_a = create_sequences(data['X_site_a'], data['y_site_a'], config.sequence_length)
    X_c, y_c = create_sequences(data['X_site_c'], data['y_site_c'], config.sequence_length)
    X_b, y_b = create_sequences(data['X_site_b'], data['y_site_b'], config.sequence_length)

    # Combine A + C for training
    X_train = np.vstack((X_a, X_c))
    y_train = np.vstack((y_a, y_c))

    X_val = X_b
    y_val = y_b

    # -------------------------------
    # 3. Build PyTorch Datasets
    # -------------------------------
    train_dataset = PowerDataset(X_train, y_train)
    val_dataset = PowerDataset(X_val, y_val)

    # Ensure val is deterministic/chronological: shuffle=False and drop_last=False
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, drop_last=False)

    # -------------------------------
    # 4. Extract metadata for metrics
    # -------------------------------
    # Inverse-transform building power for NMAE/NRMSE calculations
    train_bp_scaled = X_train[:, -1, 2]  # column index 2 = building power
    val_bp_scaled = X_val[:, -1, 2]

    # Helper array to inverse-transform only building power
    t_train = np.zeros((len(train_bp_scaled), 19))
    t_train[:, 2] = train_bp_scaled
    t_a = t_train[:len(X_a)]
    t_c = t_train[len(X_a):]

    bp_a = data['scaler_X_site_a'].inverse_transform(t_a)[:, 2]
    bp_c = data['scaler_X_site_c'].inverse_transform(t_c)[:, 2]
    train_building_power = np.concatenate([bp_a, bp_c])

    t_val = np.zeros((len(val_bp_scaled), 19))
    t_val[:, 2] = val_bp_scaled
    val_building_power = data['scaler_X_site_b'].inverse_transform(t_val)[:, 2]

    # Demand flag extraction
    train_demand_flags = np.argmax(X_train[:, -1, 30:33], axis=1) - 1
    val_demand_flags = np.argmax(X_val[:, -1, 30:33], axis=1) - 1

    # Site labels for metrics
    train_sites = np.concatenate([
        np.array(['Site A'] * len(bp_a)),
        np.array(['Site C'] * len(bp_c))
    ])
    val_sites = np.array(['Site B'] * len(X_val))

    # -------------------------------
    # 5. Initialize model + optimizer
    # -------------------------------
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = TransformerRegressor(
        config.input_size,
        config.d_model,
        config.nhead,
        config.num_layers,
        config.output_size,
        config.dropout
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )

    # -------------------------------
    # 5b. OneCycleLR scheduler (smooth)
    # -------------------------------
    steps_per_epoch = max(1, len(train_loader))
    # pct_start reflects warmup fraction: fallback to 0.1 if warmup_epochs not set or zero
    pct_start = (getattr(config, "warmup_epochs", 0) / max(1, getattr(config, "num_epochs", 1)))
    pct_start = float(np.clip(pct_start, 0.01, 0.5))  # keep pct_start reasonable
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=config.learning_rate,
        steps_per_epoch=steps_per_epoch,
        epochs=config.num_epochs,
        pct_start=pct_start,
        anneal_strategy='cos',  # smooth cosine annealing
        div_factor=25.0,        # starting LR = max_lr/div_factor
        final_div_factor=1e4    # final LR very small
    )

    # -------------------------------
    # 6. Early stopping setup
    # -------------------------------
    early_stopping_patience = getattr(config, "early_stopping_patience", 10)
    best_nmae = float('inf')
    epochs_without_improvement = 0

    print("Starting training...")

    # -------------------------------
    # 7. Training loop
    # -------------------------------
    for epoch in range(config.num_epochs):

        model.train()
        train_loss = 0.0
        preds_all = []
        targets_all = []

        # Track losses for logging
        epoch_base_losses = []
        epoch_penalties = []
        epoch_total_losses = []

        # ---------------------------
        # Training batches
        # ---------------------------
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)

            # --- CUSTOM PENALTY (no threshold version) ---
            base_loss = criterion(outputs, y_batch)

            # Identify "no DR" samples from one-hot flags
            dr_flags = X_batch[:, -1, 30:33]
            dr_ids = torch.argmax(dr_flags, dim=1)
            # mask where dr_id == 1 corresponds to "no DR" in your previous code
            mask_no_dr = (dr_ids == 1)

            if mask_no_dr.any():
                preds_no_dr = outputs[mask_no_dr].view(-1)
                penalty_term = torch.mean(preds_no_dr ** 2)
            else:
                # ensure same device and dtype
                penalty_term = torch.tensor(0.0, device=device, dtype=base_loss.dtype)

            penalty_weight = float(getattr(config, "fp_penalty_weight", 50.0))
            loss = base_loss + penalty_weight * penalty_term
            # ------------------------------------------------

            # Track for epoch averaging
            epoch_base_losses.append(base_loss.item())
            epoch_penalties.append(penalty_term.item())
            epoch_total_losses.append(loss.item())

            # Backprop
            optimizer.zero_grad()
            loss.backward()
            clip_grad_norm_(model.parameters(), getattr(config, "gradient_clip_val", 1.0))
            optimizer.step()

            # Step the scheduler per batch (smooth LR progression)
            try:
                scheduler.step()
            except Exception:
                # in edge cases (e.g., 0 steps_per_epoch) scheduler.step may fail; ignore
                pass

            train_loss += loss.item()
            preds_all.append(outputs.detach().cpu().numpy())
            targets_all.append(y_batch.cpu().numpy())

        # Average train loss across batches
        train_loss = train_loss / max(1, len(train_loader))

        # -------------------------------
        # 8. Inverse transform predictions (keep ordering consistent)
        # -------------------------------
        preds_all = np.concatenate(preds_all, axis=0)  # shape (N, 1) or (N,)
        targets_all = np.concatenate(targets_all, axis=0)

        # Ensure correct shape for scalers: (N, 1)
        if preds_all.ndim == 1:
            preds_all = preds_all.reshape(-1, 1)
        if targets_all.ndim == 1:
            targets_all = targets_all.reshape(-1, 1)

        preds_a = data['scaler_y_site_a'].inverse_transform(preds_all[:len(y_a)])
        preds_c = data['scaler_y_site_c'].inverse_transform(preds_all[len(y_a):len(y_a)+len(y_c)])
        targs_a = data['scaler_y_site_a'].inverse_transform(targets_all[:len(y_a)])
        targs_c = data['scaler_y_site_c'].inverse_transform(targets_all[len(y_a):len(y_a)+len(y_c)])

        train_preds = np.concatenate([preds_a, preds_c]).flatten()
        train_targets = np.concatenate([targs_a, targs_c]).flatten()

        # Training metrics
        train_metrics = evaluate_all_metrics(
            y_true=train_targets,
            y_pred=train_preds,
            site_labels=train_sites,
            building_power=train_building_power,
            demand_flags=train_demand_flags
        )

        # -------------------------------
        # 9. Validation (deterministic ordering preserved)
        # -------------------------------
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_targs = []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)

                loss = criterion(outputs, y_batch)
                val_loss += loss.item()

                val_preds.append(outputs.cpu().numpy())
                val_targs.append(y_batch.cpu().numpy())

        val_loss = val_loss / max(1, len(val_loader))

        val_preds = np.concatenate(val_preds, axis=0)
        val_targs = np.concatenate(val_targs, axis=0)

        if val_preds.ndim == 1:
            val_preds = val_preds.reshape(-1, 1)
        if val_targs.ndim == 1:
            val_targs = val_targs.reshape(-1, 1)

        # Inverse transform with site B scaler
        val_preds = data['scaler_y_site_b'].inverse_transform(val_preds).flatten()
        val_targs = data['scaler_y_site_b'].inverse_transform(val_targs).flatten()

        val_metrics = evaluate_all_metrics(
            y_true=val_targs,
            y_pred=val_preds,
            site_labels=val_sites,
            building_power=val_building_power,
            demand_flags=val_demand_flags
        )

        # Current LR for logging (read from optimizer)
        current_lr = optimizer.param_groups[0]['lr']

        # -------------------------------
        # 10. Log to W&B
        # -------------------------------
        wandb.log({
            "epoch": epoch,
            "learning_rate": current_lr,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/nmae_mean": train_metrics['nmae_mean'],
            "val/nmae_mean": val_metrics['nmae_mean'],
            "train/base_loss": np.mean(epoch_base_losses) if epoch_base_losses else 0.0,
            "train/penalty": np.mean(epoch_penalties) if epoch_penalties else 0.0,
            "train/total_loss": np.mean(epoch_total_losses) if epoch_total_losses else 0.0
        })

        # Print occasionally
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"\nEpoch {epoch+1}/{config.num_epochs}")
            print(f"LR: {current_lr:.6f}")
            print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            print(f"Val NMAE(mean): {val_metrics['nmae_mean']:.2f}%")

        # -------------------------------
        # 11. Early stopping + save best
        # -------------------------------
        current_val_nmae = val_metrics["nmae_mean"]

        if current_val_nmae < best_nmae:
            best_nmae = current_val_nmae
            epochs_without_improvement = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved best model at epoch {epoch+1} (Val NMAE: {best_nmae:.2f}%)")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stopping_patience:
            print(f"\nEarly stopping at epoch {epoch+1}. Best Val NMAE: {best_nmae:.2f}%")
            break

    print(f"\nTraining completed! Best Val NMAE: {best_nmae:.2f}%")
    return best_nmae


In [ ]:
if do_sweep:
    sweep_config = {
        "method": "bayes",
        "metric": {"name": "val/nmae_mean", "goal": "minimize"},
        "parameters": {
            # -----------------------------------
            # Transformer model parameters
            # -----------------------------------
            "input_size": {"value": 37},
            "d_model": {"values": [64, 128, 256]},
            "nhead": {"values": [2, 4, 8]},
            "num_layers": {"values": [2, 4, 6]},
            "output_size": {"value": 1},

            # Dropout uniform search
            "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},

            # -----------------------------------
            # Sequence length
            # -----------------------------------
            "sequence_length": {"values": [24, 48, 96, 168]},

            # -----------------------------------
            # Optimizer hyperparameters
            # -----------------------------------
            "learning_rate": {
                "distribution": "log_uniform_values",
                "min": 1e-6,
                "max": 1e-4
            },
            "weight_decay": {
                "distribution": "log_uniform_values",
                "min": 1e-8,
                "max": 1e-3
            },

            # -----------------------------------
            # Training parameters
            # -----------------------------------
            "batch_size": {"values": [16, 32, 64]},
            "num_epochs": {"value": 1000},
            "warmup_epochs": {"value": 5},

            # Gradient clipping
            "gradient_clip_val": {"distribution": "uniform", "min": 0.5, "max": 2.0},

            # Custom FP penalty
            "fp_penalty_weight": {"values": [10.0, 25.0, 50.0, 100.0]},
        }
    }

    sweep_id = wandb.sweep(
        sweep_config,
        project="AICOMP_Flextrack",
        entity="fabian-dubach-hochschule-luzern"
    )
    print(f"Sweep created: {sweep_id}")

    wandb.agent(
        sweep_id,
        function=train_model,
        count=500
    )

else:
    default_config = {
        # Model
        "input_size": 37,
        "d_model": 64,
        "nhead": 2,
        "num_layers": 2,
        "output_size": 1,
        "dropout": 0.2,

        # Data
        "sequence_length": 48,

        # Optimization
        "learning_rate": 3e-5,
        "weight_decay": 1e-4,
        "batch_size": 64,
        "num_epochs": 200,
        "warmup_epochs": 5,

        # Training behavior
        "gradient_clip_val": 1.0,
        "fp_penalty_weight": 50.0,
    }

    train_model(default_config)